# F1 Race Prediction — End-to-End Data Engineering & ML on AWS

## Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                        AWS Cloud                                  │
│                                                                  │
│  EventBridge ──► Lambda (Ingest) ──► S3: raw/season=YYYY/       │
│                                             │                    │
│                        S3 trigger ──────────┘                   │
│                             │                                    │
│                             ▼                                    │
│                   Lambda (Transform) ──► S3: processed/          │
│                                                 │                │
│                   S3: models/ ◄── train.py      │               │
│                        │                        │               │
│                        ▼                        ▼               │
│  EventBridge ──► Lambda (Predict) ──► S3: predictions/          │
│                                               │                  │
│  API Gateway ──► Lambda (LLM Q&A) ◄───────────┘                 │
│                        │                                         │
│                        ▼                                         │
│                  Claude API (tool_use)                           │
└──────────────────────────────────────────────────────────────────┘
```

## Data Source
- **Ergast API mirror**: `https://api.jolpi.ca/ergast/f1`
- Free, no auth needed, full history from 1950

## ML Targets
- `is_winner` — predict race winner (binary classification, output: probability per driver)
- `is_podium` — predict top 3 finishers (binary classification)

## This Notebook
Runs the full pipeline **locally** (no AWS needed) as an exploration and demo.
Set `USE_AWS = True` to read/write from S3 instead of local files.


In [ ]:
# ── Cell 1: Setup & Configuration ─────────────────────────────────
import os
import json
import pickle
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # pyright: ignore[reportMissingImports]
import seaborn as sns
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────────
USE_AWS        = False                   # Set True to use S3 instead of local files
S3_BUCKET      = 'f1-data-pipeline'     # Your S3 bucket name
LOCAL_DATA_DIR = Path('data')            # Local data directory when USE_AWS=False
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

TRAIN_SEASONS  = list(range(2018, 2025)) # seasons used for training
PREDICT_SEASON = 2025                    # season to predict

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_DATA_DIR / 'raw').mkdir(exist_ok=True)
(LOCAL_DATA_DIR / 'processed').mkdir(exist_ok=True)
(LOCAL_DATA_DIR / 'models').mkdir(exist_ok=True)
(LOCAL_DATA_DIR / 'predictions').mkdir(exist_ok=True)

print('Setup complete.')
print(f'Training seasons : {TRAIN_SEASONS}')
print(f'Prediction season: {PREDICT_SEASON}')
print(f'Using AWS        : {USE_AWS}')


In [ ]:
# ── Cell 2: F1 API Client ─────────────────────────────────────────
class F1APIClient:
    BASE_URL = 'https://api.jolpi.ca/ergast/f1'

    def __init__(self, timeout: int = 30):
        self.timeout = timeout
        self.session = requests.Session()

    def _get(self, endpoint: str, params: dict = None) -> dict:
        url = f'{self.BASE_URL}/{endpoint}'
        resp = self.session.get(url, params=params or {}, timeout=self.timeout)
        resp.raise_for_status()
        return resp.json()

    def get_race_results(self, season: int) -> pd.DataFrame:
        data = self._get(f'{season}/results.json', {'limit': 1000})
        rows = []
        for race in data['MRData']['RaceTable']['Races']:
            for r in race.get('Results', []):
                pos = r.get('position', '')
                rows.append({
                    'season':           int(race['season']),
                    'round':            int(race['round']),
                    'circuit_id':       race['Circuit']['circuitId'],
                    'race_name':        race['raceName'],
                    'driver_id':        r['Driver']['driverId'],
                    'constructor_id':   r['Constructor']['constructorId'],
                    'grid':             int(r.get('grid', 0) or 0),
                    'position':         int(pos) if pos.isdigit() else None,
                    'points':           float(r.get('points', 0) or 0),
                    'status':           r.get('status', ''),
                    'is_winner':        1 if pos == '1' else 0,
                    'is_podium':        1 if pos in ('1', '2', '3') else 0,
                })
        return pd.DataFrame(rows)

    def get_qualifying(self, season: int) -> pd.DataFrame:
        data = self._get(f'{season}/qualifying.json', {'limit': 1000})
        rows = []
        for race in data['MRData']['RaceTable']['Races']:
            for q in race.get('QualifyingResults', []):
                rows.append({
                    'season':       int(race['season']),
                    'round':        int(race['round']),
                    'circuit_id':   race['Circuit']['circuitId'],
                    'driver_id':    q['Driver']['driverId'],
                    'quali_pos':    int(q['position']),
                })
        return pd.DataFrame(rows)

    def get_driver_standings(self, season: int) -> pd.DataFrame:
        data = self._get(f'{season}/driverStandings.json', {'limit': 100})
        rows = []
        for sl in data['MRData']['StandingsTable']['StandingsLists']:
            rnd = int(sl.get('round', 0))
            for s in sl['DriverStandings']:
                rows.append({
                    'season':                   int(sl['season']),
                    'round':                    rnd,
                    'driver_id':                s['Driver']['driverId'],
                    'championship_position':    int(s['position']),
                    'championship_points':      float(s['points']),
                    'season_wins':              int(s['wins']),
                })
        return pd.DataFrame(rows)

    def get_schedule(self, season: int) -> pd.DataFrame:
        data = self._get(f'{season}/races.json', {'limit': 30})
        races = data['MRData']['RaceTable']['Races']
        rows = [{
            'season':       int(r['season']),
            'round':        int(r['round']),
            'race_name':    r['raceName'],
            'circuit_id':   r['Circuit']['circuitId'],
            'date':         r.get('date', ''),
        } for r in races]
        return pd.DataFrame(rows)


client = F1APIClient()
print('F1APIClient ready.')


In [ ]:
# ── Cell 3: Fetch Historical Data (one season as demo) ────────────
# Tip: for training, the Lambda ingest job fetches all TRAIN_SEASONS.
# Here we fetch just 2023 + 2024 to keep things fast interactively.

DEMO_SEASONS = [2023, 2024]

all_results, all_quali = [], []
for s in DEMO_SEASONS:
    print(f'Fetching season {s}...')
    results_df = client.get_race_results(s)
    quali_df   = client.get_qualifying(s)
    all_results.append(results_df)
    all_quali.append(quali_df)
    # Save raw locally
    results_df.to_csv(LOCAL_DATA_DIR / 'raw' / f'results_{s}.csv', index=False)
    quali_df.to_csv(LOCAL_DATA_DIR / 'raw' / f'qualifying_{s}.csv', index=False)

results = pd.concat(all_results, ignore_index=True)
quali   = pd.concat(all_quali,   ignore_index=True)

print(f'\nResults rows : {len(results):,}')
print(f'Qualifying rows: {len(quali):,}')
results.head(3)


In [ ]:
# ── Cell 4: Quick EDA ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Grid position vs wins
winner_grid = (
    results[results['is_winner'] == 1]
    .merge(quali, on=['season', 'round', 'driver_id'], how='left')
)
axes[0].hist(winner_grid['quali_pos'].dropna(), bins=20, edgecolor='black', color='#e10600')
axes[0].set_xlabel('Qualifying Position of Race Winner')
axes[0].set_ylabel('Count')
axes[0].set_title('Race Winners by Qualifying Position (2023-2024)')

# Top drivers by wins
top_drivers = (
    results[results['is_winner'] == 1]
    .groupby('driver_id').size()
    .sort_values(ascending=False)
    .head(10)
)
top_drivers.plot(kind='bar', ax=axes[1], color='#1f77b4', edgecolor='black')
axes[1].set_title('Top 10 Race Winners (2023-2024)')
axes[1].set_ylabel('Wins')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 5: Feature Engineering ──────────────────────────────────
def engineer_features(results_df: pd.DataFrame, quali_df: pd.DataFrame) -> pd.DataFrame:
    """Build ML feature matrix from raw results + qualifying data."""
    df = (
        results_df
        .merge(quali_df[['season', 'round', 'driver_id', 'quali_pos']],
               on=['season', 'round', 'driver_id'], how='left')
        .sort_values(['season', 'round'])
        .reset_index(drop=True)
    )

    # Grid position: prefer qualifying position, fallback to race grid
    df['grid_position'] = df['quali_pos'].fillna(df['grid'])

    # Rolling driver points form (last 3 and 5 races)
    for n in [3, 5]:
        df[f'driver_form_{n}'] = (
            df.groupby('driver_id')['points']
            .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
        )

    # Rolling constructor points form
    constructor_pts = (
        df.groupby(['season', 'round', 'constructor_id'])['points']
        .sum().reset_index(name='constructor_pts')
        .sort_values(['season', 'round'])
    )
    for n in [3, 5]:
        constructor_pts[f'constructor_form_{n}'] = (
            constructor_pts.groupby('constructor_id')['constructor_pts']
            .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
        )
    df = df.merge(
        constructor_pts.drop(columns='constructor_pts'),
        on=['season', 'round', 'constructor_id'], how='left'
    )

    # Historical avg finishing position at this circuit
    df['circuit_avg_pos'] = (
        df.groupby(['driver_id', 'circuit_id'])['position']
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    # Season wins accumulated before this race
    df['season_wins_so_far'] = (
        df.groupby(['driver_id', 'season'])['is_winner']
        .transform(lambda x: x.shift(1).cumsum().fillna(0))
    )

    # Races since last win (momentum proxy)
    def races_since_win(series):
        out, count = [], 0
        for v in series:
            out.append(count)
            count = 0 if v == 1 else count + 1
        return pd.Series(out, index=series.index)

    df['races_since_win'] = (
        df.groupby('driver_id')['is_winner']
        .transform(races_since_win)
    )

    # DNF rate (rolling 10 races)
    df['is_dnf'] = df['position'].isna().astype(int)
    df['dnf_rate'] = (
        df.groupby('driver_id')['is_dnf']
        .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
    )

    return df


FEATURE_COLS = [
    'grid_position',
    'driver_form_3', 'driver_form_5',
    'constructor_form_3', 'constructor_form_5',
    'circuit_avg_pos',
    'season_wins_so_far',
    'races_since_win',
    'dnf_rate',
]

features_df = engineer_features(results, quali)
print(f'Feature matrix shape: {features_df.shape}')
features_df[FEATURE_COLS + ['is_winner', 'is_podium']].head(5)


In [ ]:
# ── Cell 6: Train XGBoost Models ─────────────────────────────────
# Install if needed:  pip install xgboost scikit-learn
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

train_df = features_df.dropna(subset=FEATURE_COLS + ['is_winner', 'is_podium'])

X = train_df[FEATURE_COLS].values
y_winner = train_df['is_winner'].values
y_podium = train_df['is_podium'].values

# XGBoost handles class imbalance via scale_pos_weight
neg_pos_winner = (y_winner == 0).sum() / (y_winner == 1).sum()
neg_pos_podium = (y_podium == 0).sum() / (y_podium == 1).sum()

model_winner = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg_pos_winner,
    eval_metric='logloss', random_state=42, verbosity=0
)
model_podium = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg_pos_podium,
    eval_metric='logloss', random_state=42, verbosity=0
)

# Cross-validated AUC
cv_winner = cross_val_score(model_winner, X, y_winner, cv=5, scoring='roc_auc')
cv_podium = cross_val_score(model_podium, X, y_podium, cv=5, scoring='roc_auc')
print(f'Winner model  CV AUC: {cv_winner.mean():.3f} ± {cv_winner.std():.3f}')
print(f'Podium model  CV AUC: {cv_podium.mean():.3f} ± {cv_podium.std():.3f}')

# Fit on full dataset
model_winner.fit(X, y_winner)
model_podium.fit(X, y_podium)

# Save locally (or upload to S3 in production)
model_bundle = {'winner': model_winner, 'podium': model_podium, 'features': FEATURE_COLS}
with open(LOCAL_DATA_DIR / 'models' / 'f1_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print('Models saved to data/models/f1_model.pkl')


In [ ]:
# ── Cell 7: Feature Importance ───────────────────────────────────
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'winner_imp': model_winner.feature_importances_,
    'podium_imp': model_podium.feature_importances_,
}).sort_values('winner_imp', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title, color in [
    (axes[0], 'winner_imp', 'Winner Model Feature Importance', '#e10600'),
    (axes[1], 'podium_imp', 'Podium Model Feature Importance', '#1f77b4'),
]:
    ax.barh(importance_df['feature'], importance_df[col], color=color, edgecolor='black')
    ax.set_title(title)
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: Predict Upcoming Race ────────────────────────────────
def predict_next_race(season: int, round_num: int | None = None) -> pd.DataFrame:
    """
    Predict winner & podium for the next (or specified) race.
    Uses latest qualifying results + driver rolling stats.
    """
    # Load model
    with open(LOCAL_DATA_DIR / 'models' / 'f1_model.pkl', 'rb') as f:
        bundle = pickle.load(f)
    m_winner = bundle['winner']
    m_podium = bundle['podium']
    feature_cols = bundle['features']

    # Fetch all current season results + qualifying
    print(f'Fetching {season} data...')
    season_results = client.get_race_results(season)
    season_quali   = client.get_qualifying(season)
    schedule       = client.get_schedule(season)

    if season_results.empty:
        print('No race results yet for this season — predictions based on historical stats only.')

    # Identify next race
    if round_num is None:
        completed_rounds = set(season_results['round'].unique()) if not season_results.empty else set()
        all_rounds = set(schedule['round'].unique())
        upcoming = sorted(all_rounds - completed_rounds)
        if not upcoming:
            print('Season complete — showing last race predictions.')
            round_num = max(all_rounds)
        else:
            round_num = upcoming[0]

    race_info = schedule[schedule['round'] == round_num].iloc[0]
    print(f'\nPredicting: Round {round_num} — {race_info["race_name"]} ({race_info["circuit_id"]})')

    # Get qualifying for this round
    round_quali = season_quali[season_quali['round'] == round_num]
    if round_quali.empty:
        print('No qualifying data yet. Using last available qualifying order.')
        last_round = season_quali['round'].max() if not season_quali.empty else 1
        round_quali = season_quali[season_quali['round'] == last_round].copy()
        round_quali['round'] = round_num

    # Build historical feature stats per driver
    # Combine prior seasons + current season results for rolling stats
    hist_all = pd.concat([features_df, engineer_features(season_results, season_quali)], ignore_index=True)
    hist_all = hist_all.sort_values(['season', 'round'])

    # Latest stats per driver
    driver_stats = (
        hist_all.drop_duplicates(subset='driver_id', keep='last')
        [['driver_id', 'driver_form_3', 'driver_form_5',
          'constructor_id', 'constructor_form_3', 'constructor_form_5',
          'season_wins_so_far', 'races_since_win', 'dnf_rate']]
    )

    # Circuit avg pos at this specific circuit
    circuit_id = race_info['circuit_id']
    circuit_stats = (
        hist_all[hist_all['circuit_id'] == circuit_id]
        .drop_duplicates(subset='driver_id', keep='last')
        [['driver_id', 'circuit_avg_pos']]
    )

    # Assemble prediction frame
    pred_df = (
        round_quali[['driver_id', 'quali_pos']].rename(columns={'quali_pos': 'grid_position'})
        .merge(driver_stats, on='driver_id', how='left')
        .merge(circuit_stats, on='driver_id', how='left')
    )
    pred_df.fillna(pred_df.median(numeric_only=True), inplace=True)

    X_pred = pred_df[feature_cols].values
    pred_df['win_prob']    = m_winner.predict_proba(X_pred)[:, 1]
    pred_df['podium_prob'] = m_podium.predict_proba(X_pred)[:, 1]

    pred_df = pred_df.sort_values('win_prob', ascending=False).reset_index(drop=True)
    pred_df.index += 1

    return pred_df, race_info


predictions, race_info = predict_next_race(PREDICT_SEASON)
print(f'\n── Predictions: {race_info["race_name"]} ──')
print(predictions[['driver_id', 'grid_position', 'win_prob', 'podium_prob']].head(10).to_string())


In [ ]:
# ── Cell 9: Visualize Predictions ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top10 = predictions.head(10)

axes[0].barh(top10['driver_id'][::-1], top10['win_prob'][::-1], color='#e10600', edgecolor='black')
axes[0].set_xlabel('Win Probability')
axes[0].set_title(f'Race Winner Probability — {race_info["race_name"]}')
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.5)

axes[1].barh(top10['driver_id'][::-1], top10['podium_prob'][::-1], color='#1f77b4', edgecolor='black')
axes[1].set_xlabel('Podium Probability')
axes[1].set_title(f'Podium Probability — {race_info["race_name"]}')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print('\nPredicted Podium:')
for i, row in predictions.head(3).iterrows():
    print(f'  P{i}: {row["driver_id"]:20s}  win={row["win_prob"]:.2%}  podium={row["podium_prob"]:.2%}')


In [ ]:
# ── Cell 10: LLM Q&A with Claude ─────────────────────────────────
# Requires: pip install anthropic
# Set ANTHROPIC_API_KEY environment variable
import anthropic

# Prepare prediction context for tools
_predictions_json = predictions.head(10)[['driver_id', 'grid_position', 'win_prob', 'podium_prob']].to_dict(orient='records')
_race_context = {'race_name': race_info['race_name'], 'circuit': race_info['circuit_id'], 'season': PREDICT_SEASON}

# Define tools Claude can call
TOOLS = [
    {
        'name': 'get_race_predictions',
        'description': 'Get ML model predictions for the upcoming F1 race (win probability and podium probability per driver).',
        'input_schema': {
            'type': 'object',
            'properties': {
                'top_n': {'type': 'integer', 'description': 'How many top drivers to return (default 10)'}
            },
            'required': []
        }
    },
    {
        'name': 'get_race_info',
        'description': 'Get information about the upcoming race (name, circuit, season).',
        'input_schema': {'type': 'object', 'properties': {}, 'required': []}
    },
    {
        'name': 'get_driver_form',
        'description': 'Get recent form stats for a specific driver.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'driver_id': {'type': 'string', 'description': 'The Ergast driver ID (e.g. verstappen, hamilton)'}
            },
            'required': ['driver_id']
        }
    }
]

def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    if tool_name == 'get_race_predictions':
        n = tool_input.get('top_n', 10)
        return json.dumps(_predictions_json[:n])
    elif tool_name == 'get_race_info':
        return json.dumps(_race_context)
    elif tool_name == 'get_driver_form':
        did = tool_input['driver_id'].lower()
        match = predictions[predictions['driver_id'].str.lower() == did]
        if match.empty:
            return json.dumps({'error': f'Driver {did} not found in predictions'})
        row = match.iloc[0]
        return json.dumps({
            'driver_id': did,
            'grid_position': int(row['grid_position']),
            'driver_form_3': float(row['driver_form_3']),
            'driver_form_5': float(row['driver_form_5']),
            'win_prob':    float(row['win_prob']),
            'podium_prob': float(row['podium_prob']),
        })
    return json.dumps({'error': 'Unknown tool'})


def ask_f1_llm(question: str) -> str:
    """Ask Claude a question about the F1 race predictions."""
    anth = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    messages = [{'role': 'user', 'content': question}]

    system = (
        'You are an expert F1 analyst with access to ML model predictions for the upcoming race. '
        'Use the available tools to fetch prediction data before answering. '
        'Be specific with probabilities and explain the key factors driving your analysis.'
    )

    while True:
        response = anth.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=1024,
            system=system,
            tools=TOOLS,
            messages=messages
        )

        if response.stop_reason == 'end_turn':
            return ''.join(b.text for b in response.content if hasattr(b, 'text'))

        # Handle tool use
        messages.append({'role': 'assistant', 'content': response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                result = handle_tool_call(block.name, block.input)
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': result
                })
        messages.append({'role': 'user', 'content': tool_results})


# Demo questions
questions = [
    'Who is predicted to win the next race and why?',
    'Who are the predicted top 3 finishers?',
]

if ANTHROPIC_API_KEY:
    for q in questions:
        print(f'\nQ: {q}')
        print(f'A: {ask_f1_llm(q)}')
        print('─' * 60)
else:
    print('Set ANTHROPIC_API_KEY to enable LLM Q&A.')
    print('The llm_qa Lambda (f1_pipeline/lambdas/llm_qa/handler.py) does the same via API Gateway.')


## AWS Deployment Guide

### 1. Provision Infrastructure
```bash
cd f1_pipeline/infrastructure
pip install boto3
python aws_setup.py
```

### 2. Deploy Lambda Functions
Each Lambda lives in `f1_pipeline/lambdas/<name>/handler.py`.
Package with dependencies and upload:
```bash
cd f1_pipeline/lambdas/ingest
pip install requests -t .
zip -r ingest.zip .
aws lambda update-function-code --function-name f1-ingest --zip-file fileb://ingest.zip
```

### 3. EventBridge Schedule (ingest runs Friday before race weekend)
```
cron(0 12 ? * FRI *)   → f1-ingest Lambda
cron(0 16 ? * SAT *)   → f1-predict Lambda  (after qualifying)
```

### 4. LLM Q&A API
- POST `https://<api-gw>.execute-api.<region>.amazonaws.com/prod/ask`
- Body: `{"question": "Who will win the next race?"}`

### 5. Train & Upload Model
```bash
python f1_pipeline/ml/train.py --seasons 2018 2019 2020 2021 2022 2023 2024 --upload
```
